In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / "src"))

import pandas as pd
import config 
import pipeline as pl

In [2]:
estimate_full = pl.estimate_all(
    countries=list(config.SEA_COUNTRIES),
    bboxes=[config.SEA_BBOX],
    start=config.STUDY_START,
    end=config.STUDY_END,
)

    source  requests  transactions  estimated_gb  estimated_minutes                                                      note
     FIRMS       439          2192         0.022                4.4 1 box around 11 countries x 2192 days, 5 days per request
 ERA5-Land        73            73       206.220             2190.0          196,000 grid points x 52,608 hours x 5 variables
WorldCover       238           238        10.472              119.0        238 distinct tiles (windowed reads fetch far less)
     TOTAL       750          2503       216.710             2313.4                                                          

Estimated wall clock: 38.6 hours (1.6 days)
Estimated disk      : 216.7 GB


In [3]:
estimate_b = pl.estimate_all(
    countries=config.STUDY_COUNTRIES,
    bboxes=list(config.STUDY_REGIONS.values()),
    start=config.STUDY_START,
    end=config.STUDY_END,
)

    source  requests  transactions  estimated_gb  estimated_minutes                                                     note
     FIRMS       439          2192         0.022                4.4 1 box around 5 countries x 2192 days, 5 days per request
 ERA5-Land       146           146         1.260             4380.0           1,200 grid points x 52,608 hours x 5 variables
WorldCover         8             8         0.352                4.0         8 distinct tiles (windowed reads fetch far less)
     TOTAL       593          2346         1.630             4388.4                                                         

Estimated wall clock: 73.1 hours (3.0 days)
Estimated disk      : 1.6 GB


In [4]:
comparison = pd.DataFrame({
    "Original plan": estimate_full[estimate_full["source"] == "TOTAL"].iloc[0],
    "Reduced scope": estimate_b[estimate_b["source"] == "TOTAL"].iloc[0],
}).loc[["requests", "estimated_gb", "estimated_minutes"]]

comparison["Reduction factor"] = (
    comparison["Original plan"] / comparison["Reduced scope"]
).round(1)
comparison

,Original plan,Reduced scope,Reduction factor
requests,750,593,1.264755
estimated_gb,216.71,1.63,132.95092
estimated_minutes,2313.4,4388.4,0.527163


In [5]:
scenarios = {
    "Original: 11 countries, all of Southeast Asia, 6 years": (list(config.SEA_COUNTRIES), [config.SEA_BBOX],
                                        "2020-01-01", "2025-12-31"),
    "B: 5 countries, 2 regions, 6 years":      (config.STUDY_COUNTRIES, list(config.STUDY_REGIONS.values()),
                                        "2020-01-01", "2025-12-31"),
    "D: 5 countries, 2 regions, 2 years":      (config.STUDY_COUNTRIES, list(config.STUDY_REGIONS.values()),
                                        "2024-01-01", "2025-12-31"),
    "E: 5 countries, 1 region, 2 years":      (config.STUDY_COUNTRIES, [config.BBOX_CHIANG_MAI],
                                        "2024-01-01", "2025-12-31"),
}

rows = []
for name, (c, b, s, e) in scenarios.items():
    est = pl.estimate_all(c, b, s, e, verbose=False)
    total = est[est["source"] == "TOTAL"].iloc[0]
    rows.append({
        "Scenario": name,
        "Request": int(total["requests"]),
        "GB": total["estimated_gb"],
        "Hours": round(total["estimated_minutes"] / 60, 1),
    })

pd.DataFrame(rows)

,Scenario,Request,GB,Hours
0,"Original: 11 countries, all of Southeast Asia,...",750,216.71,38.6
1,"B: 5 countries, 2 regions, 6 years",593,1.63,73.1
2,"D: 5 countries, 2 regions, 2 years",203,0.78,24.1
3,"E: 5 countries, 1 region, 2 years",175,0.39,12.1


In [6]:
import logging
pl.setup_logging(level="WARNING")

calls = {"n": 0}

def flaky_network(x):
    calls["n"] += 1
    if calls["n"] < 3:
        raise ConnectionError("Network connection lost")
    return "Success"

result = pl.call_with_retry(flaky_network, {"x": 1}, base_delay=0.5)
print(result, "- after", calls["n"], "attempts")

15:51:57  WARNING  flaky_network failed (attempt 1/3): Network connection lost - retrying in 0s
15:51:57  WARNING  flaky_network failed (attempt 2/3): Network connection lost - retrying in 1s


Success - after 3 attempts


In [7]:
calls2 = {"n": 0}

def invalid_parameter(x):
    calls2["n"] += 1
    raise ValueError("Incorrect bounding box coordinate order")

try:
    pl.call_with_retry(invalid_parameter, {"x": 1}, base_delay=0.5)
except ValueError as e:
    print(f"Stopped immediately after {calls2['n']} attempt(s): {e}")

Stopped immediately after 1 attempt(s): Incorrect bounding box coordinate order


In [8]:
for start, days in pl.split_into_chunks("2025-03-01", "2025-03-12", chunk_days=5):
    print(f"{start}  +{days} days")

print()
print("Total number of chunks for the entire project, for one country:", len(pl.split_into_chunks(config.STUDY_START, config.STUDY_END, 5)))

2025-03-01  +5 days
2025-03-06  +5 days
2025-03-11  +2 days

Total number of chunks for the entire project, for one country: 439


In [ ]:
manifest = pl.read_manifest()
print(f"Number of records in the manifest: {len(manifest)}")

if len(manifest) > 0:
    display(pl.manifest_summary())
    display(manifest.tail(5))
else:
    print("(download_all.py has not been run yet)")